### Scripts for Queries in "The Alvenhime Assassin: A DnD WhoDunit"
#### A Ren'py game built by Chahna Ahuja and Liangyu Gan for ISI Course, 2025-26
<br>

*SOME NOTES ON THE FILE:*

- **All additional comments and code information are written in markdown to format and systemize the Jupyter Notebook.**
  
- **Input statements will not be shown as outputs in PDF (a shortfall for exporting as PDF), so the input statements are mentioned in code comments**
  
- **Table of Contents**
    1. [**Imports and User Info for MYSQL**](#imports)
    2. [**Importing SQL Database and Connecting Engine**](#engine)
    3. [**Query Builder**](#builder) 
    4. [**Aggregate Class**](#aggregate)
    5. [**Query Execution and JSON Record**](#execute) 
    6. [**Queries for Project:**](#project)
        1) [*Scene 2-4: Finding Suspected Items (Queries 1 to 7)*](#scene2-4)
           1. [Scene 2: Poison Trail Branch](#scene2)
           2. [Scene 3: Guard Weapon Branch](#scene3)
           3. [Scene 4: Noble Poison Weapon Branch](#scene4)
        2) [*Scene 5-6: Analyzing Customer Records and Identifying Assassin (Queries 8 to 11)*](#scene5-6)
           1. [Scene 5: Right Ending](#scene5)
           2. [Scene 6: Wrong Ending](#scene6)

<a id="imports"></a>
### 1. Imports and User Info for MySQL

The following imports need to run before initiating any other code. 
- `PANDAS` used for reading SQL queries and convert the results into JSON
- `JSON` to load, write and dump queries in JSON file.
- `SQLALCHEMY` and other modules in it are used to connect Jupyter with Phpmyadmin and SQL database in it, initalize and read the queries.

The user info for mysql **should be modified** depending upon the machine where the code is running as it is dependent on user's phpmyadmin. 

In [ ]:
import pandas as pd
import json
import os
import sqlalchemy
from sqlalchemy import text, create_engine
from sqlalchemy_utils import database_exists, create_database

In [ ]:
#user info for connecting to MYSQL in PhpmyAdmin using Xampp, you can change it if it is different on your pc
user='root'
host = "localhost"
port = 3306
password="" 

<a id="engine"></a>

### 2. Importing SQL Database and Connecting Engine

Before running the below code block in this section, please go to Project Archive → Database → Import **aaa_shop.sql** in your own PhpMyAdmin so you already have this database for creating and connecting the engine from the sqlalchemy module. Enter the database name is given as input to make this more accessible for the reader. 

In [ ]:
dbname=input("Enter the database name: ") #database is aaa_shop, please write this!

In [ ]:
engine = create_engine(f'mysql+mysqlconnector://{user}:{password}@{host}:{port}/{dbname}') #create engine
if not database_exists(engine.url): #this is just a safe measure, but will not be needed if you already import database
    create_database(engine.url)  
conn = engine.connect() #connect engine 

<a id="builder"></a>

### 3. Query Builder 

> **Satisifies the requirement of avoiding hard-coded queries as much as possible by constructing queries dynamically through functions in a class**

#### Class Functions for MYSQL Functions

We have attempted our best to ensure that this query builder handles all simple and complex queries! To explain the functions it supports: 

- **want_column variable:** It either takes string or list of columns to collects all column in a list to retrieve columns a user desires.
  
- **table_name variable:** variable that stores the table name from which the query is retrieved.

  
- **join list variable:** a list to store all joins of a query.

  
- **filters dictionary variable:** a dictionary that has where, group by and having as keys that stores their respective conditions as values.

  
- **add_case function:** a function to implement CASE statement in Python. It has a when_then_dict, a dictionary to store conditions and else_outcome stores the else clause. Alias is just to add any column aliases if a value is returned on conditions being met. *(a function not taught in class)*

  
- **add_left_join function**: left join function select data from the left table and match it with every corresponding row from right table; takes   table_name and on clause as parameters and appends to a list of joins.

  
- **add_join function**: To join tables based on a related column between them; takes table_name and on clause as parameters and appends to a list of joins.

  
- **add_where function**: Takes column, value and operators as arguments to write where conditions and append to the key 'where' in filters dictionary.

  
- **add_where_raw function:** this was an absurd but neccessary function to create because different operators like between functions couldn't be used in add_where. So this just appends raw string conditions of where to the dictionary.

  
- **add_where_in:** Similar to add_where but created explicitly for where conditions that use IN as operator.

  
- **add_group_by function:** Takes one or multiple columns as arguments, stores in key 'group_by* in filters dictionary to perform the group_by clause.

  
- **add_having function:** Takes aggregate variable, operator variable, value variable and raw variable(exception cases) as arguments. We defined to ways to add having clause, either by having a combination of aggregate +operator+value or just using the raw string. Customized for the project requirements

- **set_order_by function:** Takes clause and sort as paremeters where default is ascending sort.

- **set_limit function:** Takes limit string as a paremeter to set a limit statement in MySQL

#### Building SQL String 

**build function**: The `build` function brings everything together by concatenating all parts into a MySQL-readable query. It follows this order:

- Join all columns in `want_columns` as a string, separated by commas.
- Write the `SELECT` statement with the table name provided by the user.
- For the `WHERE` clause:
  - Initialize an empty `conditions` list to store all `WHERE` conditions.
  - In the `where` key of the `filters` dictionary, conditions are stored as strings:
    1. A tuple of value and operator (default operator is `=`).
    2. A raw string to deal with where in and other complex conditions not handled by the query builder).
  - Collect all conditions in the list and join them with `" AND "` (OR conditions are not used, as they were not required).
  - Append the resulting string to the main query as the `WHERE` clause.
- For the `GROUPBY` clause, all group_by columns are concatenated with a comma
- For the `HAVING` clause, same logic is applied! (OR conditions not accounted for, as not required for our queries)
- `ORDER BY` and `LIMIT` are also concatenated

>**Please Note:** *To build this query builder, we actually could not find any inspiration. Following simple functions to do dynamic queries in Exercise session and dynamic querybuilder discussion in class, we took an adhoc approach by making a querybuilder that suit our requirements. We just coded this with our own logic and refined it as needed with queries.*

In [ ]:
class QueryBuilder:
    def __init__(self, table_name, want_column="*"):
        self.table = table_name #table name variable
        # Handle different types of want_column, multiple as list and as string
        if isinstance(want_column, list): 
            self.columns = []
            for col in want_column: 
                self.columns.append(str(col))
        else:
            self.columns=[]
            col_str = str(want_column)           
            self.columns.append(col_str) 

        self.joins = [] #empty list for joins
        #filters dictionary for WHERE, GROUPBY, HAVING
        self.filters = {"where": {}, "group_by": [], "having": []} 
        self.order_by = None #orderby default none
        self.limit = None #limit default none

    def add_case_column(self, when_then_dict, else_outcome, alias):
        parts = ["CASE"]
        #when then dictionary as input- when is key and then is value
        for condition, result in when_then_dict.items(): 
            parts.append(f"    WHEN {condition} THEN {result}")
        #else outcome
        parts.append(f"    ELSE {else_outcome}") 
        #optional alias
        parts.append(f"END AS {alias}")
        #case expression together
        case_expr = "\n".join(parts)  
        #append as column
        self.columns.append(case_expr) 
        return self
        
    def add_left_join(self, table_name, on_clause):
        #specify table and condition for relating columns
        self.joins.append(f"LEFT JOIN {table_name} ON {on_clause}")  
        return self

    def add_join(self, table_name, on_clause):
        #specify table and condition for relating columns 
        self.joins.append(f"JOIN {table_name} ON {on_clause}") 
        return self

    def add_where(self, column, value, operator="="):
        # specify column, value and operator
        self.filters["where"][column] = (value, operator) 
        return self

    def add_where_raw(self, condition):
        #raw string - for wherein and other complex conditions
        self.filters["where"][f"raw_{len(self.filters['where'])}"] = (condition, "RAW") 
        return self
    
    def add_where_in(self, column, values):
        quoted_values = [] #which values are in the column (more than one usually, so a list)
        for val in values:
            if isinstance(val, str):
                quoted_values.append(f"'{val}'") 
            else:
                quoted_values.append(str(val))
        in_clause = f"{column} IN ({', '.join(quoted_values)})" 
        
        self.filters["where"][f"{column}_in"] = (in_clause, "RAW")
        return self

    def add_group_by(self, *columns):
        #option to add more than one columns
        self.filters["group_by"].extend(columns)  
        return self

    def add_having(self, aggregate=None, operator=None, value=None, raw=None):

        if raw:
            self.filters["having"].append(raw) #raw option 1
        # aggregate option 2, specify aggregate, operator and value
        elif aggregate and operator and value is not None: 
            if not isinstance(aggregate, str):
                agg_str = str(aggregate)   # convert non-string to string
            else:
                agg_str = aggregate #if already string            
            self.filters["having"].append(f"{agg_str} {operator} {value}")
        else:
            #just a safety measure, as hard to find this error usually
            raise ValueError("Provide either raw SQL or aggregate + operator + value") 
        return self


    def set_order_by(self, clause,sort="ASC"):
        #convert sort clause always in upper
        self.order_by = f"{str(clause)} {sort.upper()}" 
        return self

    def set_limit(self, limit):
        self.limit = limit #limit as argument 
        return self

    def build(self):
        #join want columns with comma 
        columns_str = ", ".join(self.columns) 
        #select statement
        query = f"SELECT {columns_str} FROM {self.table}" 
        if self.joins: #join
            query += " " + " ".join(self.joins) 
        if self.filters["where"]: #where  
            #initializing list to store conditions 
            conditions = [] 
            for col, val_op in self.filters["where"].items():
                if isinstance(val_op, tuple):
                    #different operators 
                    val, op = val_op 
                else: #default op is =
                    val, op = val_op, "=" 
    
                op_upper = op.upper() 
                #if condition is RAW like in where_in
                if op_upper == "RAW": 
                    conditions.append(val)
    
                else:
                #if string with col, op, val
                    if isinstance(val, str):
                        val = f"'{val}'"
                    #append to list
                    conditions.append(f"{col} {op} {val}")
    
            # Join all conditions into the WHERE and AND
            query += " WHERE " + " AND ".join(conditions)
            
        if self.filters["group_by"]: 
            query += " GROUP BY " + ", ".join(self.filters["group_by"]) #groupby 
        if self.filters["having"]:
            query += " HAVING " + " AND ".join(self.filters["having"]) #having 

        if self.order_by:
            query += f" ORDER BY {self.order_by}" #orderby 
        if self.limit:
            query += f" LIMIT {self.limit}" #Limit

        return query


<a id="aggregate"></a>

### 4. Aggregate class

> **Satisifies the requirement of avoiding hard-coded queries as much as possible by constructing queries dynamically through functions in a class**

This is a class that can be called for aggregate functions. By making a class for aggregates, we made an attempt to make even adding aggregates in query dynamic! It supports:
- **_generate_expression (main function)**: Internal helper to format the SQL expressions with aggregates, like ensuring aggregate in upper case, adding alias
- **count_all function**: `COUNT (*)` aggregate, alias default None. Uses generate_expression to return outcome.
- **count_distinct function**: `COUNT (DISTINCT)` aggregate, alias default None. Does not use generate_expression here because it does not currently support DISTINCT inside the parentheses. *(an adhoc way because we made this function later for a specific query)*
- **max function**: `MAX` aggregate, alias default None. Uses generate_expression to return outcome.
- **min function**: `MIN` aggregate, alias default None. Uses generate_expression to return outcome.
- **sum function**: `SUM` aggregate, alias default None. Uses generate_expression to return outcome.
- **avg function**: `AVG` aggregate`, alias default None. Uses generate_expression to return outcome.

In [ ]:
class Aggregate:

    def _generate_expression(function_name, column, alias=None):
        # Ensure the function name is uppercase
        expression = f"{function_name.upper()}({column})"
    
        if alias: #adding alias
            expression += f" AS {alias}"
            
        return expression

    def count_all(alias=None):
        return Aggregate._generate_expression('COUNT', '*', alias)
        
    def count_distinct(column, alias=None):
        expression = f"COUNT(DISTINCT {column})"
        if alias:
            expression += f" AS {alias}"
        return expression

    def max(column, alias=None):
        return Aggregate._generate_expression('MAX', column, alias)

    def min(column, alias=None):
        return Aggregate._generate_expression('MIN', column, alias)

    def sum(column, alias=None):
        return Aggregate._generate_expression('SUM', column, alias)

    def average(column, alias=None):
        return Aggregate._generate_expression('AVG', column, alias)

<a id="execute"></a>

### 5. Query Execution and JSON Record 

> **Satisifies the requirement of avoiding hard-coded queries as much as possible by constructing queries dynamically through functions in a class**
> **Satisfies the requirement of executing the script and converting the results into a JSON file**

`execute_query` function is a function to run the query builder for buiding SQL readable string, read the sql query using pandas and convert the results into json. 

> **PLEASE NOTE**: *The function asks for the json file path as input and loads the query data into json. We kept it user input because if the user wants to load this into a JSON file that is not in the Renpy folder (as the current database.json will already exist in Renpy games folder), they can do that as well. Secondly, we want to input for every query so when we run all cells all queries don't run at once (another shortfall for printing Jupyter notebook to PDF that requires to run all cells at once for PDF conversion)*



In [ ]:
def execute_query(qb, record_name, write_file=True):
    # Build SQL
    if isinstance(qb, QueryBuilder):
        sql_query = qb.build()
    else:
        sql_query = qb

    # Run query
    df = pd.read_sql(sql_query, conn)
    result = df.to_json(orient="records")

    # Convert to Python object
    new_data = {record_name: json.loads(result)}

    #User input of file path 
    user_input_path = input("Enter the full path for the JSON file")
    #for us, filepath is 
    #"E:\ISI_Final_Project\game\database.json"
    file_path = os.path.normpath(user_input_path)  
    

    
    if write_file:
        # Load existing JSON if it exists
        if os.path.exists(file_path):
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    existing_data = json.load(f)
                except json.JSONDecodeError:
                    existing_data = {}
        else:
            existing_data = {}

        # Merge new data
        existing_data.update(new_data)

        # Write back to file
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(existing_data, f, indent=4)

        # Print full JSON so you can see it in Jupyter
        print(existing_data)

        return existing_data
    
    # If write_file=False, return the new data
    print(new_data)
    return new_data

<a id="project"></a>

### 6. Queries for Project 

Following code cells are about the queries used in every scene in Renpy. We briefly also elaborate the scene so the reader of this file can follow along the Renpy Game. 

>This section satifies the following assignment requirements: 
*(1) Writing down what each query does,*
*(2) Writing down the query in SQL in a Python script,*
*(3) Also mentions what query requirements as per the assignment it satifies.*

**This is a game flowchart to help you follow along the different scenes to better understand where queries are used!**

In [ ]:
from IPython.display import Image, display

img_path = r"E:\Jupyter Programs\ISI_Group Project\game_flowchart.png"

display(Image(filename=img_path))

<a id="scene2-4"></a>

### Scene 2-4: Finding Identifying Items

In the first four scenes of the game. The player as Detective has to find:
- **Poison that killed Noble (Scene 2)**
- **Weapon that killed Guard (Scene 3)**
- **Weapon through which poison was administered in Noble's body (Scene 4)**

<a id="scene2"></a>

#### Scene 2: Poison Trail Branch (Queries 1 to 3)

Scene 2 'Poison Trail' branch is about finding the poison that killed the noble. As the player proceeds to Scene 2. The fae shopkeeper asks whether they know anything about this poison. There are two choices: 
**(1) I know poison type**
**(2) I know poison potency.**



##### **Query 1: I know poison type! Poison Type is Injury**

- This query pertains to the first choice if the player chooses `I know poison type`. 
- Then, the player is asked to choose what type? 
    - If the player chooses poison type as `Injury` *(given as a clue in Scene 1)*, this query is used to give results of suspected **injury** poisons. 
    - *(Other poison types selection generate an error, and prompts try again*)


In [ ]:
#Query 1: Poison_Type is Injury 

qb = QueryBuilder(table_name="Products p", want_column="p.name")
qb.add_join("Poisons po", "p.id = po.id")
qb.add_where("po.type", "Injury")

record_json=execute_query(qb,"poison_injury")


##### **Query 2: I know poison potency!**



- This query pertains to the second choice if the player chooses `I know poison potency` 
- Then the player is asked to input the potency level as a number (int format).
    - If the player types `12` as potency level, this query is used to give results to give results of suspected **12 potency level** poisons.
    - *(Other potency levels written in input take one player life, and force the user to first pick the poison type)*

*(Note this choice is usually selected when player replays this level because they find this potency level after failing investigation report of Royal Alchemist Lab and moving to Scene 3, after Scene 3, they are again asked if they wish to play this path)*

In [ ]:
# Query 2: Poison_Potency is 12 

qb = QueryBuilder(table_name="Products p", want_column="p.name")
qb.add_join("Poisons po", "p.id = po.id")
qb.add_where("po.potency",12)

record_json=execute_query(qb,"poison_potency")


##### **Query 3: After Royal Alchemist Lab's Report: Poison Found!**

- After choosing `I know poison type` and selecting `Injury`, the player is asked to roll an investigation dice check to further investigate based on the Royal Alchemist Lab Report.
- On the success of the dice roll, the player gets to know `Poison potency as 12` and `Key ingredient in Poison as Snake fang residue`.
- This leads to this query, where the exact poison **`Serpent Venom`** is found.

- *(On failing the dice roll, the player only gets to know `Poison potency as 12` and the query that gives this result is Query 2. So, we are not counting that as a new query)*

In [ ]:
#Query 3: Poison_Potency is 12 and Ingredient is Snake Fang Residue


qb = QueryBuilder(table_name="Products p", want_column="p.name")
qb.add_join("Poisons po", "p.id = po.id")
qb.add_join("pingredients pi","p.id = pi.poison_id")
qb.add_where("pi.ingredient","Snake fang residue")

record_json=execute_query(qb,"p1_found")


<a id="scene3"></a>

#### Scene 3: Guard Weapon Branch (Queries 4 and 5)

In Scene 3, the player has to find the weapon that killed the guard:
- First, the player is asked to select the proficiency of the weapon used to kill the guard (Correct Anwer is `Martial`). *(given as clue in Scene 1)*
- Then, the player is asked to select the range of the weapon used to kill the guard (Correct Answer is `Ranged`). *(given as clue in Scene 1)*



##### **Query 4: Count of Martial Ranged Weapons**

> **Satifies the requirement of at least one query that contains an aggregation function**

This query is used when Fae shopkeeper gives the number of Martial Ranged weapons in the store.

In [ ]:
#Query 4 Count all weapons that are martial and ranged 

count_all= Aggregate.count_all(alias="total")
qb = QueryBuilder(table_name="Weapons w", want_column=count_all)
qb.add_where("w.proficiency","Martial")
qb.add_where("w.combat","Ranged")

record_json=execute_query(qb,"count_suspectguardweapon")


##### **Query 5: Weapon that killed guard found!**

- This query generate results when the player **passes the charisma dice check** or **successfully plays the mini game** to find the exact weapon.
- After that, the Detective player gets a hint about the Ammunition of the weapon.
- Then, the player finally finds the weapon the `Martial Ranged Weapon with Ammunition (30/120)`


In [ ]:
#Query 5 Guard weapon found

qb = QueryBuilder(table_name="Weapons w", want_column="p.name")
qb.add_join("Products p","w.id=p.id")
qb.add_where("w.proficiency","Martial")
qb.add_where("w.combat","Ranged")
qb.add_where("w.properties","%Ammunition (30/120)%", operator="LIKE")

record_json=execute_query(qb,"p2_found")



<a id="scene4"></a>

#### Scene 4: Noble Poison Weapon Branch (Queries 6 and 7)

In Scene 4, the player has to find the weapon through which the poison was injected in the noble. 
- First, the player is asked about the wound caused by this weapon. (Correct Answer: `Piercing`, Wrong Answer: Retry) *(mentioned as clue in Scene 1)*
- Then, the player is asked whether this weapon was **thrown** or **loaded** (Correct Answer `Thrown`). Wrong answer costs one player life. *(This is a guesswork question)*
- After that, the player is given an additional information that the weapon is very light, `less than 3 lb`.

##### **Query 6 Minimum weight of suspected weapon that poisoned noble**

> **Satifies the requirement of at least one query that contains an aggregation function with HAVING**

This query generates results when the sly Fae shopkeeper only gives the weight of the weapon inquired by the player. 

In [ ]:
#Query 6 Minimum weight of suspected weapon through which noble was poisoned

want_column=Aggregate.min("w.weight_lb","suspectpoisonweapon")
aggregate=Aggregate.min("w.weight_lb")
qb = QueryBuilder(table_name="Weapons w", want_column=want_column)
qb.add_where("w.damage","Piercing")
qb.add_where("w.properties","%Thrown%", operator="LIKE")
qb.add_group_by("w.damage")
qb.add_having(aggregate=aggregate,operator="<",value=3)

record_json=execute_query(qb,"suspectpoisonweapon")


##### **Query 7: Suspected weapon for poison delivery found!**

This query gives the suspected weapon for poison delivery when the player successfully passes the perception dice check. *(On failing dice check, player has to try again)*

In [ ]:
#Query 7 Noble poison weapon found

qb = QueryBuilder(table_name="Weapons w", want_column="p.name")
qb.add_join("Products p","w.id=p.id")
qb.add_where("w.weight_lb",0.25)

record_json=execute_query(qb,"p3_found")

<a id="scene5-6"></a>

### Scene 5-6 Analyzing Customer Records and Identifying Assassin

In the last two scenes of the game, the player who has found either all or most of suspected items now needs to analyze customer records at the shop.

- They find the assassin in **Scene 5 Right Ending** *(Provided player reaches here)*
- They have to guess the assassin in **Scene 6 Wrong Ending** *(As the player is not sure about the poison)*

<a id="scene5"></a>

#### Scene 5: Right Ending (Queries 8 and 9)

After finding `Serpent Venom`, `Crossbow hand` and `Dart` as the correct suspected items. The player finds themselves in Scene 5, which is the **Right Ending**. 

- First, the player finds suspect who bought all suspected items.
- Then, they get more hints to find the assassin. 

##### **Query 8: Identify Suspects with correct suspected items**

Who bought `Serpent Venom`, `Crossbow hand` and `Dart` from the shop?

In [ ]:
#Query 8 Suspects if all three items recovered

aggregate=Aggregate.count_distinct("p.name")
qb = QueryBuilder("sales s", "c.name")
qb.add_join("customers c","s.customer_id=c.id")
qb.add_join("products p","s.product_id=p.id")
qb.add_where_in("p.name", ["Dart", "Crossbow hand", "Serpent venom"])
qb.add_group_by("c.name")
qb.add_having(aggregate, "=", 3)
record_json=execute_query(qb,"suspects")

##### **Query 9: Assassin Found!**

> **Satifies the requirement of using JOIN in 3 tables**

- After the player narrows down suspects, they go through an intimidation dice check *(bound to pass this as threshold is designed that way)*
- They get two more hints from the Fae shopkeeper
    - `Serpent venom` must be bought in **high quantities**, close to 1000 portions to poison a noble.
    - `Serpent venom` must be used **within six months** to be a potent poison for killing.
- They get one last clue from the Royal Secret Services that suspect belongs to **Rogue** class and they must match the customer suspect with their own suspect list.

So, this query is about **finding the customer who bought max serpent poison bought within six months of noble's death in October 2023. They also bought Darts, Crossbow hand. And most importantly, they are a rogue and should be on Secret Services's suspect list.**

In [ ]:
#Query 9:Finding killer right ending who has max serpent poison bought in 2023 
#along with darts,crossbow hand and is on royal court's rogue suspect list 

suspect_names = [d['name'] for d in record_json['suspects']]
    
year="YEAR(s.sale_date) AS `Purchase Year of Dart, Serpent Venom and Hand Crossbow`"
month="MONTHNAME(s.sale_date) AS `Purchase Month of Dart, Serpent Venom and Hand Crossbow`"
aggregate = Aggregate.max("s.quantity", alias="`Serpent Venom Quantity`")
want_column=["c.name","c.class",year,month,aggregate]
qb = QueryBuilder("sales s",want_column=want_column)
qb.add_join("customers c","s.customer_id=c.id")
qb.add_join("products p","s.product_id=p.id")
qb.add_join("slist l","l.name=c.name")
qb.add_where("p.name","Serpent Venom")
qb.add_where_raw("s.sale_date >= '2023-05-01'") 
qb.add_where_raw("s.sale_date < '2023-10-31'") 
qb.add_where_in("c.name",suspect_names)
qb.add_group_by("c.id","c.name","c.class")
qb.set_order_by("`Serpent Venom Quantity`",sort="DESC")
qb.set_limit(1)


record_json=execute_query(qb,"killer")


<a id="scene6"></a>

#### Scene 6: Wrong Ending (Queries 10 and 11)

The player reaches in Scene 6 if they know suspected items are `Dart`, `Crossbow hand` but are not sure about the poison as it could either be: `Serpent venom` or `Drow poison`. 

- So, they are asked to guess which poison they think is the most probable: **Drow Poison** or **Serpent Venom**.

*(For Drow poison, a new query has to be written. For Serpent venom option, Query 8 is reused)*

- Then, they get one last hint to find the assassin.

##### **Query 10: Suspects who bought Drow Poison, Crossbow Hand and Darts**

Who bought `Drow Poison`, `Crossbow hand` and `Darts` from shop?

In [ ]:
#Query 10 Drow Poison,Crossbow Hand, Dart suspects

aggregate=Aggregate.count_distinct("p.name")
qb = QueryBuilder("sales s", "c.name")
qb.add_join("customers c","s.customer_id=c.id")
qb.add_join("products p","s.product_id=p.id")
qb.add_where_in("p.name", ["Dart", "Crossbow hand", "Drow poison"])
qb.add_group_by("c.name")
qb.add_having(aggregate, "=", 3)
record_json=execute_query(qb,"wrong_suspect")

##### **Query 11: Guess the Assassin**

> **Satifies the requirement of using a MYSQL function not taught in class. We used CASE statement**

- After the player narrows down suspects depending upon what they think is the suspected poison, they are given one last clue from the Royal Secret Services that suspect belongs to **Rogue** class and they must match the customer suspect with their own suspect list.
- So, the three suspects are now: 1. Marnia Blackstrand 2. Mecretia Coldshore 3. Roksana Roxley *(first two bought Serpent venom along with other suspected items and last one bought Drow poison along with other suspected items)*
- Depending upon what the player encountered as suspects when they chose their suspected poison, they now only can guess the assassin from these suspects that must match as a **Rogue** who exists in Royal Secret Services' suspect list.

Whatever, they select, the outcome will tell them if their guess is right or wrong. 

In [ ]:
#Query 11 Check if any of these names exist in Royal Court's suspect list


qb = QueryBuilder("customers c", want_column=["c.name","c.class"])
qb.add_left_join("slist l", "c.name = l.name")  # left join to keep all suspects
qb.add_case_column(
    when_then_dict={
        "c.name = 'Mecretia  Coldshore'": "'You are correct'",
        "c.name = 'Marnia  Blackstrand'": "'You are wrong'",
        "c.name = 'Roksana  Roxley'": "'You are wrong'"
    },
    else_outcome="NULL",
    alias="outcome"
)
qb.add_where_in("c.name", ["Mecretia  Coldshore","Marnia  Blackstrand","Roksana  Roxley"])


record_json=execute_query(qb,"last_guess")